#### Simple sequential LLM workflow

START --> LLM QA --> END 

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace # add HF_TOKEN in .env file
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
# create the model
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)
model.invoke("what is the capital of India")

d:\youtube\youtube-udemy\ai_engineering\code\langgraph\.lg-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 41, 'total_tokens': 50}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c0382-44b8-7330-b66d-19bc5cdf13a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 9, 'total_tokens': 50})

In [4]:
# 2. Define the state 
class LLMState(TypedDict):
    question: str 
    answer: str 

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    question = state['question']
    prompt = f"Answer the following {question}"

    content = model.invoke(prompt).content 
    state['answer'] = content
    return state 


In [6]:
graph = StateGraph(LLMState)

graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

workflow = graph.compile()

In [7]:
initial_state = {'question':'What is the capital of India'}
final_state = workflow.invoke(initial_state)
print(final_state)


{'question': 'What is the capital of India', 'answer': 'The capital of India is New Delhi.'}
